In [7]:
from google.colab import files
uploaded = files.upload()



Saving Dataset .csv to Dataset .csv


In [9]:
df = pd.read_csv("Dataset .csv")

In [10]:
df.head()


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [11]:
df = df[['Restaurant Name', 'Cuisines', 'City', 'Price range', 'Aggregate rating', 'Votes']]
df.head()

,Restaurant Name,Cuisines,City,Price range,Aggregate rating,Votes
0,Le Petit Souffle,"French, Japanese, Desserts",Makati City,3,4.8,314
1,Izakaya Kikufuji,Japanese,Makati City,3,4.5,591
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",Mandaluyong City,4,4.4,270
3,Ooma,"Japanese, Sushi",Mandaluyong City,4,4.9,365
4,Sambo Kojin,"Japanese, Korean",Mandaluyong City,4,4.8,229


In [12]:
df['Cuisines'] = df['Cuisines'].fillna('Unknown')
df['City'] = df['City'].fillna('Unknown')


In [13]:
encoded_features = pd.get_dummies(df[['Cuisines', 'City', 'Price range']])


In [14]:
numeric_features = df[['Aggregate rating', 'Votes']]

feature_matrix = pd.concat([encoded_features, numeric_features], axis=1)
feature_matrix.head()

,Price range,Cuisines_Afghani,"Cuisines_Afghani, Mughlai, Chinese","Cuisines_Afghani, North Indian","Cuisines_Afghani, North Indian, Pakistani, Arabian",Cuisines_African,"Cuisines_African, Portuguese",Cuisines_American,"Cuisines_American, Asian, Burger","Cuisines_American, Asian, European, Seafood",...,City_Vineland Station,City_Vizag,City_Waterloo,City_Weirton,City_Wellington City,City_Winchester Bay,City_Yorkton,City_��stanbul,Aggregate rating,Votes
0,3,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,4.8,314
1,3,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,4.5,591
2,4,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,4.4,270
3,4,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,4.9,365
4,4,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,4.8,229


In [15]:
similarity_matrix = cosine_similarity(feature_matrix)

In [16]:
def recommend_restaurants(cuisine, city, price_range, top_n=5):

    # Filter restaurants matching user preference
    filtered_df = df[
        (df['Cuisines'].str.contains(cuisine, case=False)) &
        (df['City'].str.lower() == city.lower()) &
        (df['Price range'] == price_range)
    ]

    if filtered_df.empty:
        return "No matching restaurants found."

    # Get index of first matching restaurant
    idx = filtered_df.index[0]

    # Get similarity scores
    sim_scores = list(enumerate(similarity_matrix[idx]))

    # Sort by similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top recommendations (excluding itself)
    sim_scores = sim_scores[1:top_n+1]

    restaurant_indices = [i[0] for i in sim_scores]

    return df.iloc[restaurant_indices][['Restaurant Name', 'Cuisines', 'City', 'Aggregate rating']]

In [17]:
recommend_restaurants("Italian", "New Delhi", 2)

,Restaurant Name,Cuisines,City,Aggregate rating
5857,Chicago Pizza,"Fast Food, Italian, Pizza",New Delhi,3.4
3012,MOB Brewpub,"Continental, Italian, Asian, Indian",New Delhi,4.7
3258,Cafe Yell,"Cafe, Continental, Italian",New Delhi,4.4
2572,Nariyal Cafe,"Continental, Seafood, Goan, Andhra, Kerala, Thai",New Delhi,4.2
6844,Cafe Hashtag LoL,"Cafe, North Indian, Chinese",New Delhi,4.2
